In [1]:
import numpy as np
import pandas as pd

def build_input_dict(
    charger_ids: list[str],
    value: float | dict[str, float],
) -> dict[str, float]:
    """
    Build a per-charger input dictionary from either a scalar or an
    existing dict."""

    if isinstance(value, dict):
            return value                                    # already per-charger
    return {cid: value for cid in charger_ids}         # broadcast scalar → dict

In [2]:
# from pyspark.sql import SparkSession

# # Create a Spark Session with Databricks
# spark = SparkSession.builder.getOrCreate()

# # ── Time Index ─────────────────────────────────────────────────────────────────
# start_time = pd.Timestamp('2025-01-01')
# end_time   = pd.Timestamp('2025-01-31')
# idx        = pd.date_range(start=start_time, freq='15min', end=end_time)

# # Pre-format timestamps as SQL-safe strings once — reuse everywhere
# start_str = start_time.strftime('%Y-%m-%d %H:%M:%S')   # '2026-01-01 00:00:00'
# end_str   = end_time.strftime('%Y-%m-%d %H:%M:%S')     # '2026-04-01 00:00:00'

# charger_ids     = ['ECTX6MQM', 'EC93FYLG', 'ECR5S7HB', 'EC4BSFH6']
# charger_ids_sql = ", ".join(f"'{cid}'" for cid in charger_ids)

# # ── Load raw data from Databricks ─────────────────────────────────────────────
# sql_raw = f"""
#     SELECT
#         carConnected,
#         carDisconnected,
#         kiloWattHours,
#         chargerId,
#         SiteName
#     FROM ckw_gbed_dev.flex_bronze.tbl_easee_sessions
#     WHERE chargerId    IN ({charger_ids_sql})
#       AND carConnected >  '{start_str}'
# """

# df  = spark.sql(sql_raw)
# pdf = df.toPandas()     # bring subset to local memory

# # ── Query 2 — Dynamic Tariffs ──────────────────────────────────────────────────
# sql_dyn_tariff = f"""
#     SELECT start_timestamp_UTC, CHF_per_kWh
#     FROM ckw_gbed_dev.flex_silver.tbl_dynamic_tariffs
#     WHERE tariff_name           =  'CKW_grid_home_Rp_per_kWh'
#       AND start_timestamp_UTC  >= TIMESTAMP('{start_str}')
#       AND start_timestamp_UTC  <= TIMESTAMP('{end_str}')
# """

# df_2 = spark.sql(sql_dyn_tariff)
# df_dyn_tarif = df_2.toPandas()


In [ ]:
import numpy as np
import pandas as pd

# ── Session Data ──────────────────────────────────────────────────────────────
easee_sessions = pd.read_parquet(r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\01_raw\filtered_easee_sessions.parquet')
site_data = pd.read_parquet(r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\01_raw\filtered_easee_sites.parquet')
easee_sessions['carConnected']    = pd.to_datetime(easee_sessions['carConnected'], utc=True)
easee_sessions['carDisconnected'] = pd.to_datetime(easee_sessions['carDisconnected'], utc=True)

# ── Sessions per site ID with min/max carConnected ────────────────────────────
sessions_per_site = (
    easee_sessions
    .groupby('site_id')
    .agg(
        session_count = ('site_id',      'size'),
        first_session = ('carConnected', 'min'),
        last_session  = ('carConnected', 'max'),
    )
    .reset_index()
    .sort_values('session_count', ascending=False)
)

sessions_per_site = sessions_per_site.merge(
    site_data[['id']].drop_duplicates(),
    left_on='site_id', right_on='id',
    how='left'
).drop(columns='id')

print(sessions_per_site[['site_id', 'session_count', 'first_session', 'last_session']].to_string(index=False))

pdf = easee_sessions[easee_sessions['site_id'] == 275947]

# ── Price Data ──────────────────────────────────────────────────────────────
df_dyn_tarif = pd.read_csv(r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\01_raw\home_dynamic_Leistungspreis_CKW_25.csv')
df_dyn_tarif['Timestamp'] = pd.to_datetime(df_dyn_tarif['Timestamp'], dayfirst=True, errors='coerce')
df_dyn_tarif['Timestamp'] = df_dyn_tarif['Timestamp'].dt.tz_localize('Europe/Zurich', ambiguous='infer')
df_dyn_tarif['Timestamp'] = df_dyn_tarif['Timestamp'].dt.tz_convert('UTC')
df_dyn_tarif = df_dyn_tarif.set_index('Timestamp', drop=True).sort_index()

# ── Time Index ─────────────────────────────────────────────────────────────────
start_time = pd.Timestamp('2025-01-01', tz='UTC')
end_time   = pd.Timestamp('2026-01-01', tz='UTC')
idx        = pd.date_range(start=start_time, freq='15min', end=end_time, tz='UTC')

# ── Optimization Dataframe -────────────────────────────────────────────────────
columns      = ['power_min', 'power_max', 'e_in', 'e_out', 'e_cap']
df_ev_inputs = pd.DataFrame(index=idx, columns=columns, dtype=float)
df_ev_inputs.index = pd.to_datetime(df_ev_inputs.index, format='mixed', dayfirst=True)

# ── Assign to df_ev_inputs ─────────────────────────────────────────────────────
df_ev_inputs['dyn_tarif'] = df_dyn_tarif['home dynamic']
df_ev_inputs['dyn_tarif'] = df_ev_inputs['dyn_tarif'] / 100 # From Rp to CHF 
df_ev_inputs['dyn_tarif'] = df_ev_inputs['dyn_tarif'].ffill()

In [4]:
def build_expensive_hour_mask(
    df: pd.DataFrame,
    price_col: str,
    n_hours: int,
) -> pd.Series:
    """Return a single 0/1 mask Series blocking the n most expensive hours/day."""
    if not isinstance(df.index, pd.DatetimeIndex):
        raise KeyError("DataFrame must have a DatetimeIndex.")

    day = df.index.date
    hour = df.index.hour

    hourly_price = (
        df.groupby([day, hour])[price_col]
        .mean()
        .rename_axis(['day', 'hour'])
        .reset_index()
    )

    expensive_hours = (
        hourly_price
        .groupby('day')
        .apply(lambda g: set(g.nlargest(n_hours, price_col)['hour']))
    )

    mask = pd.Series(
        [0 if h in expensive_hours.loc[d] else 1 for d, h in zip(day, hour)],
        index=df.index,
        name=f'{price_col}_mask_{n_hours}',
    )
    return mask


In [ ]:
from hems_resopt.utils.pre_process_ev import prepare_sessions, build_connection_df, get_power_bounds, get_e_in_out_capacity
battery_sizes = 80
power_max_per_charger = 11
EV_opt_name='EV_Fleet'
columns      = ['power_min', 'power_max', 'e_in', 'e_out', 'energy_capacity']

# ── Assemble df_ev_inputs ──────────────────────────────────────────────────────

# Step 1 — prepare sessions
sessions = prepare_sessions(pdf, start_time, end_time)
# sessions = sessions.drop(columns=['Unnamed: 0'])
sessions = sessions.drop_duplicates()

# give same inputs to every charger -> alterantively a manual dict can be build with indiviudal values per chargerId
charger_ids = sessions['chargerId'].unique().tolist()   # or pass your explicit list

battery_dict = build_input_dict(charger_ids, battery_sizes)
power_dict   = build_input_dict(charger_ids, power_max_per_charger)

# Step 2 — build the central connection matrix
connection_df = build_connection_df(sessions, idx, charger_ids)

# Step 3 — fill each input via its dedicated function
df_ev_inputs[['power_min', 'power_max']] = get_power_bounds(connection_df, power_dict)
df_ev_inputs[['e_in', 'e_out', 'e_cap']], capped_kWh_list = get_e_in_out_capacity(sessions, connection_df, idx, battery_dict, power_dict)

mask_blocked_hours = build_expensive_hour_mask(
    df_ev_inputs,
    price_col='dyn_tarif',
    n_hours=4,
)

df_ev_inputs['power_min'] = df_ev_inputs['power_min'] * mask_blocked_hours

In [ ]:
import pyomo.environ as pyo
from hems_resopt.components.grid import GridPeakShave
from res_opt_core import EnergyModel, Grid, AuctionMarket, plot_battery_operation
from hems_resopt.components.ev import EV
from pyomo.contrib.solver.solvers.highs import Highs


solver = pyo.SolverFactory("highs")
solver.options["limits/time"] = 60
 
custom_solver = Highs()
#scip_config = ResOptSolverConfig(absolute_optimality_gap = 0.1)

# Loosen feasibility & optimality tolerances
custom_solver.highs_options = {
    "primal_feasibility_tolerance": 1e-4,   # default: 1e-7  (loosen)
    "dual_feasibility_tolerance":   1e-4,   # default: 1e-7  (loosen)
    "ipm_optimality_tolerance":     1e-4,   # interior point tolerance
    "time_limit":                   300.0,  # seconds — increase if needed
    "presolve":                     "on",   # keep presolve active
    "solver":                       "simplex",  # or "ipm" for interior point
}

model = EnergyModel(
    num_steps=len(df_ev_inputs.index),
    slot_length="15min",
    solver=custom_solver,
    timestamp=df_ev_inputs.index
)
 
# Components
ev_fleet = EV(
    name=EV_opt_name,
    power_nominal=200,
    power_min=df_ev_inputs['power_min'],
    power_max=df_ev_inputs['power_max'],
    energy_capacity=df_ev_inputs['e_cap'],
    soc_initial=0.0,
    soc_final=None,
    energy_in_slot_start=df_ev_inputs['e_in'],
    energy_out_slot_end=df_ev_inputs['e_out'],
    # Make the HARD soc_min/soc_max bounds temporarily soft to diagnose infeasibility
    cost_min_energy_violation=100,   # CHF/kWh - high, so only used if truly forced
    cost_max_energy_violation=100,
)



dynamischer_tariff = AuctionMarket(
    name='dynamic_tariff',
    price_curve=df_ev_inputs['dyn_tarif'],
    #market_time_unit='1hr' definier resolutoin of market 
)

# grid = Grid(
#    name="grid",
#    assets=[ev_fleet],
#    markets=[dynamischer_tariff],
    
# )

grid = GridPeakShave(
     name='Grid_with_peakshave',
     power_min=df_ev_inputs['power_min'].min()-100,
     power_max=0,
     assets=[ev_fleet],
     markets=[dynamischer_tariff],
     peak_power_price=1.5, #CHF/kW
     time_horizon_peak='month_daylight_saving'
 )

model.add_component(ev_fleet)
model.add_component(dynamischer_tariff)
model.add_component(grid)
model.build_and_run(silent=False) #scip outputs visible

df_results = model.results.timeseries_to_pandas()

In [ ]:
plot_battery_operation(df_results.index, ev_fleet, {"dynamischer_tariff": df_ev_inputs['dyn_tarif']})

In [ ]:
from hems_resopt.utils.post_process_ev import plot_charger_usage, plot_representative_week, compute_ev_optimization_summary, print_summary
ladetarif_ckw = 0.33 # CHF/kWh total alles inklusive, energie, netz, lastpitzenkosten deckung, 
lastspitzekoste_ckw = 0.0587 # CHF/kWh (Annahme Verteilung Lastspitzenkosten: bei 6000kWh jährlich und monatlicher Spitzenleistung von 14.7 kW)
peak_power_price_dyn = 1
peak_power_price_stat = 1.5

df_post_process, summary_dict, monthly_df = compute_ev_optimization_summary(
    df_results=df_results,
    df_ev_inputs=df_ev_inputs,
    idx=idx,
    sessions=sessions,
    capped_kWh_list=capped_kWh_list,
    peak_power_price_dyn=peak_power_price_dyn,
    peak_power_price_stat=peak_power_price_stat,
    energy_costs=0.11,
    fix_costs= 0.07226,
    EV_opt_name=EV_opt_name
)
print_summary(summary_dict)
plot_charger_usage(easee_sessions, df_post_process, idx, connection_df, sessions, session_kWh_col='kiloWattHours')
plot_representative_week(df_ev_inputs['dyn_tarif'], df_post_process['energy_charged'])